In [1]:
import pandas as pd
city_bike = pd.read_csv('../Data/Divvy_2024_All_Months_Cleaned.csv')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Load your data
df = city_bike.copy()

print("=" * 80)
print("BIKE SHARE PREDICTION MODEL")
print("=" * 80)

# ============================================================================
# 1. FEATURE ENGINEERING
# ============================================================================
print("\n1. FEATURE ENGINEERING")
print("-" * 80)

# Convert datetime columns
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

# Time-based features
df['hour'] = df['started_at'].dt.hour
df['day'] = df['started_at'].dt.day
df['month'] = df['started_at'].dt.month
df['year'] = df['started_at'].dt.year
df['day_of_week_num'] = df['started_at'].dt.dayofweek
df['week_of_year'] = df['started_at'].dt.isocalendar().week

# Cyclical encoding for time features (captures circular nature)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week_num'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week_num'] / 7)

# Distance calculation (Haversine formula)
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df['distance_km'] = haversine_distance(
    df['start_lat'], df['start_lng'], 
    df['end_lat'], df['end_lng']
)

# Is round trip (same start and end station)
df['is_round_trip'] = (df['start_station_id'] == df['end_station_id']).astype(int)

# Peak hours (7-9 AM, 5-7 PM)
df['is_rush_hour'] = df['hour'].apply(
    lambda x: 1 if (7 <= x <= 9) or (17 <= x <= 19) else 0
)

# Weekend indicator
df['is_weekend'] = df['day_of_week_num'].apply(lambda x: 1 if x >= 5 else 0)

# Season (for Northern Hemisphere)
def get_season(month):
    if month in [12, 1, 2]:
        return 0  # Winter
    elif month in [3, 4, 5]:
        return 1  # Spring
    elif month in [6, 7, 8]:
        return 2  # Summer
    else:
        return 3  # Fall

df['season'] = df['month'].apply(get_season)

print(f"✓ Total features: {len(df.columns)}")

# ============================================================================
# 2. TARGET VARIABLE CREATION
# ============================================================================
print("\n2. TARGET VARIABLE")
print("-" * 80)

# Primary target: ride_length (already exists)
# Alternative targets you could create:
# - Binary: is_long_ride (rides > 30 minutes)
df['is_long_ride'] = (df['ride_length'] > 30).astype(int)

# - Categorical: ride_duration_category
def categorize_ride(length):
    if length < 10:
        return 0  # Short
    elif length < 30:
        return 1  # Medium
    else:
        return 2  # Long

df['ride_category'] = df['ride_length'].apply(categorize_ride)

print("Available target variables:")
print("  - ride_length (continuous): Predict exact ride duration")
print("  - is_long_ride (binary): Predict if ride > 30 min")
print("  - ride_category (multiclass): Predict short/medium/long")

# ============================================================================
# 3. PREPARE DATA FOR MODELING
# ============================================================================
print("\n3. DATA PREPARATION")
print("-" * 80)

# Select target (using ride_length for regression)
target = 'ride_length'
y = df[target].copy()

# Select features
feature_cols = [
    'rideable_type_code', 'member_casual_code',
    'hour', 'day', 'month', 'year', 'day_of_week_num', 'week_of_year',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
    'distance_km', 'is_round_trip', 'is_rush_hour', 'is_weekend', 'season',
    'start_lat', 'start_lng', 'end_lat', 'end_lng'
]

X = df[feature_cols].copy()

# Handle missing values
X = X.fillna(X.median())

# Remove outliers from target (optional, helps model performance)
Q1 = y.quantile(0.25)
Q3 = y.quantile(0.75)
IQR = Q3 - Q1
outlier_mask = (y >= Q1 - 1.5 * IQR) & (y <= Q3 + 1.5 * IQR)
X = X[outlier_mask]
y = y[outlier_mask]

print(f"✓ Features selected: {len(feature_cols)}")
print(f"✓ Samples after outlier removal: {len(X)}")
print(f"✓ Target variable: {target}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✓ Train set: {len(X_train)} samples")
print(f"✓ Test set: {len(X_test)} samples")

# ============================================================================
# 4. COMPARE MULTIPLE MODELS
# ============================================================================
print("\n4. MODEL COMPARISON (Base Models)")
print("-" * 80)

models = {
    'Ridge Regression': Ridge(random_state=42),
    'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=10),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

model_results = {}

for name, model in models.items():
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred = model.predict(X_test_scaled)
    
    # Metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Cross-validation score
    cv_scores = cross_val_score(model, X_train_scaled, y_train, 
                                cv=5, scoring='r2', n_jobs=-1)
    
    model_results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'CV_R2_mean': cv_scores.mean(),
        'CV_R2_std': cv_scores.std()
    }
    
    print(f"\n{name}:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  R² Score: {r2:.4f}")
    print(f"  CV R² (5-fold): {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Find best model
best_model_name = max(model_results, key=lambda k: model_results[k]['R2'])
print(f"\n{'=' * 80}")
print(f"BEST BASE MODEL: {best_model_name}")
print(f"{'=' * 80}")

# ============================================================================
# 5. HYPERPARAMETER TUNING
# ============================================================================
print("\n5. HYPERPARAMETER TUNING (on best models)")
print("-" * 80)

# Tune Random Forest
print("\nTuning Random Forest...")
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_params,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)
rf_grid.fit(X_train_scaled, y_train)

print(f"Best RF parameters: {rf_grid.best_params_}")
print(f"Best RF CV score: {rf_grid.best_score_:.4f}")

# Tune Gradient Boosting
print("\nTuning Gradient Boosting...")
gb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5],
    'subsample': [0.8, 0.9, 1.0]
}

gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    gb_params,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)
gb_grid.fit(X_train_scaled, y_train)

print(f"Best GB parameters: {gb_grid.best_params_}")
print(f"Best GB CV score: {gb_grid.best_score_:.4f}")

# ============================================================================
# 6. FINAL MODEL EVALUATION
# ============================================================================
print("\n6. FINAL TUNED MODEL COMPARISON")
print("-" * 80)

tuned_models = {
    'Tuned Random Forest': rf_grid.best_estimator_,
    'Tuned Gradient Boosting': gb_grid.best_estimator_
}

final_results = {}

for name, model in tuned_models.items():
    y_pred = model.predict(X_test_scaled)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    final_results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    
    print(f"\n{name}:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  R² Score: {r2:.4f}")

# Select final best model
final_best = max(final_results, key=lambda k: final_results[k]['R2'])
best_model = tuned_models[final_best]

print(f"\n{'=' * 80}")
print(f"FINAL BEST MODEL: {final_best}")
print(f"R² Score: {final_results[final_best]['R2']:.4f}")
print(f"RMSE: {final_results[final_best]['RMSE']:.4f} minutes")
print(f"MAE: {final_results[final_best]['MAE']:.4f} minutes")
print(f"{'=' * 80}")

# ============================================================================
# 7. FEATURE IMPORTANCE
# ============================================================================
print("\n7. FEATURE IMPORTANCE (Top 10)")
print("-" * 80)

if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(feature_importance.head(10).to_string(index=False))

# ============================================================================
# 8. MAKING PREDICTIONS
# ============================================================================
print("\n8. EXAMPLE PREDICTIONS")
print("-" * 80)

# Make predictions on test set
sample_predictions = pd.DataFrame({
    'Actual': y_test.head(10).values,
    'Predicted': best_model.predict(X_test_scaled[:10]),
    'Error': y_test.head(10).values - best_model.predict(X_test_scaled[:10])
})

print(sample_predictions.to_string(index=False))

print("\n" + "=" * 80)
print("MODEL TRAINING COMPLETE!")
print("=" * 80)
print("\nTo use this model for predictions:")
print("  1. Save the model: import joblib; joblib.dump(best_model, 'bike_model.pkl')")
print("  2. Save the scaler: joblib.dump(scaler, 'scaler.pkl')")
print("  3. Load and predict: model = joblib.load('bike_model.pkl')")
print("=" * 80)

BIKE SHARE PREDICTION MODEL

1. FEATURE ENGINEERING
--------------------------------------------------------------------------------
✓ Total features: 37

2. TARGET VARIABLE
--------------------------------------------------------------------------------
Available target variables:
  - ride_length (continuous): Predict exact ride duration
  - is_long_ride (binary): Predict if ride > 30 min
  - ride_category (multiclass): Predict short/medium/long

3. DATA PREPARATION
--------------------------------------------------------------------------------
✓ Features selected: 23
✓ Samples after outlier removal: 387885
✓ Target variable: ride_length
✓ Train set: 310308 samples
✓ Test set: 77577 samples

4. MODEL COMPARISON (Base Models)
--------------------------------------------------------------------------------

Ridge Regression:
  RMSE: 5.1746
  MAE: 3.5121
  R² Score: 0.5732
  CV R² (5-fold): 0.5702 (+/- 0.0026)

Decision Tree:
  RMSE: 4.8431
  MAE: 3.1200
  R² Score: 0.6262
  CV R² (5-fo